# Imports

In [0]:
from pyspark.sql.functions import trim, when, length, lit, col, row_number, lower as lower_spark, concat_ws, coalesce, current_timestamp, sha2
from pyspark.sql.window import Window
from delta.tables import DeltaTable


In [0]:
from pyspark.sql.functions import sum as sum_spark, lower as lower_spark, upper as upper_spark, countDistinct, first, regexp_replace

In [0]:
CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

RACE_RESULTS_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.race_results"
QUALIFYING_RESULTS_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.qualifying_results"

SILVER_CONSTRUCTORS_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.constructors"

# Metodos

In [0]:
def clean_string(column_name: str):
    value = trim(col(column_name))

    return (
        when(length(value) == 0, lit(None).cast("string"))
         .otherwise(value)
    )

In [0]:
def clean_hex_color(column_name: str):
    value = clean_string(column_name)

    return upper_spark(
        regexp_replace(
            value,
            "^#",
            ""
        )
    )

In [0]:
def extract_constructor_candidates(
    table_name: str,
    source_priority: int
):
    return (
        spark.table(table_name)
        .select(
            col("season")
                .cast("int")
                .alias("_season"),

            col("round")
                .cast("int")
                .alias("_round"),

            lower_spark(
                clean_string("TeamId")
            ).alias("constructor_id"),

            clean_string("TeamName")
                .alias("constructor_name"),

            clean_hex_color("TeamColor")
                .alias("team_color_hex"),

            col("_source_entity")
                .alias("source_entity"),

            col("_source_file")
                .alias("source_file"),

            col("_source_file_modification_time")
                .alias("source_modified_at"),

            col("_ingested_at")
                .alias("bronze_ingested_at"),

            col("_rescued_data"),

            lit(source_priority)
                .alias("_source_priority")
        )
    )

## Se une de las 2 tablas

* race_result
* qualifying_result

In [0]:
constructor_candidates_df = (
    extract_constructor_candidates(
        RACE_RESULTS_TABLE,
        source_priority=2
    )
    .unionByName(
        extract_constructor_candidates(
            QUALIFYING_RESULTS_TABLE,
            source_priority=1
        )
    )
)

## Primeras validaciones

In [0]:
errors = {}

rescued_count = (
    constructor_candidates_df
    .filter(col("_rescued_data").isNotNull())
    .count()
)

if rescued_count > 0:
    errors["rescued_data"] = rescued_count

null_constructor_id = (
    constructor_candidates_df
    .filter(col("constructor_id").isNull())
    .count()
)

if null_constructor_id > 0:
    errors["null_constructor_id"] = null_constructor_id

null_constructor_name = (
    constructor_candidates_df
    .filter(col("constructor_name").isNull())
    .count()
)

if null_constructor_name > 0:
    errors["null_constructor_name"] = null_constructor_name

name_conflicts = (
    constructor_candidates_df
    .groupBy(
        "_season",
        "_round",
        "constructor_id"
    )
    .agg(
        countDistinct("constructor_name")
            .alias("name_count")
    )
    .filter(col("name_count") > 1)
    .count()
)

if name_conflicts > 0:
    errors["constructor_name_conflicts"] = name_conflicts

invalid_colors = (
    constructor_candidates_df
    .filter(
        col("team_color_hex").isNotNull()
        &
        ~col("team_color_hex").rlike("^[0-9A-F]{6}$")
    )
    .count()
)

if invalid_colors > 0:
    errors["invalid_team_color"] = invalid_colors

if errors:
    raise ValueError(
        f"Constructor candidate validation failed: {errors}"
    )

print("Constructor candidate validation OK.")

# Transformaciones

In [0]:
ordering = (
    Window
    .partitionBy("constructor_id")
    .orderBy(
        col("_season").desc(),
        col("_round").desc(),
        col("_source_priority").desc(),
        col("source_modified_at").desc_nulls_last(),
        col("bronze_ingested_at").desc_nulls_last()
    )
)

full_window = ordering.rowsBetween(
    Window.unboundedPreceding,
    Window.unboundedFollowing
)

constructor_candidates_df = (
    constructor_candidates_df

    .withColumn(
        "_canonical_constructor_name",
        first(
            "constructor_name",
            ignorenulls=True
        ).over(full_window)
    )

    .withColumn(
        "_canonical_team_color_hex",
        first(
            "team_color_hex",
            ignorenulls=True
        ).over(full_window)
    )

    .withColumn(
        "_row_number",
        row_number().over(ordering)
    )

    .filter(col("_row_number") == 1)

    .select(
        "constructor_id",

        col("_canonical_constructor_name")
            .alias("constructor_name"),

        col("_canonical_team_color_hex")
            .alias("team_color_hex"),

        "source_entity",
        "source_file",
        "source_modified_at",
        "bronze_ingested_at"
    )
)

In [0]:
constructor_columns = [
    "constructor_id",
    "constructor_name",
    "team_color_hex"
]

hash_expression = concat_ws(
    "||",
    *[
        coalesce(
            col(column).cast("string"),
            lit("<NULL>")
        )
        for column in constructor_columns
    ]
)

constructor_candidates_df = (
    constructor_candidates_df
    .withColumn(
        "record_hash",
        sha2(
            hash_expression,
            256
        )
    )
    .withColumn(
        "silver_updated_at",
        current_timestamp()
    )
)

# Ultimas validaciones

In [0]:
validation = (
    constructor_candidates_df
    .agg(
        sum_spark(
            when(
                col("constructor_id").isNull(),
                1
            ).otherwise(0)
        ).alias("null_constructor_id"),

        sum_spark(
            when(
                col("constructor_name").isNull(),
                1
            ).otherwise(0)
        ).alias("null_constructor_name")
    )
    .first()
    .asDict()
)

errors = {
    rule: value or 0
    for rule, value in validation.items()
    if (value or 0) > 0
}

duplicate_count = (
    constructor_candidates_df
    .groupBy("constructor_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

if duplicate_count > 0:
    errors["duplicate_constructor_id"] = duplicate_count

invalid_colors = (
    constructor_candidates_df
    .filter(
        col("team_color_hex").isNotNull()
        &
        ~col("team_color_hex").rlike("^[0-9A-F]{6}$")
    )
    .count()
)

if invalid_colors > 0:
    errors["invalid_team_color"] = invalid_colors

if errors:
    raise ValueError(
        f"Silver constructors validation failed: {errors}"
    )

print(
    f"Validation OK: {constructor_candidates_df.count()} constructors ready for Silver."
)

# Merge

In [0]:
if not spark.catalog.tableExists(SILVER_CONSTRUCTORS_TABLE):
    target = DeltaTable.forName(
        spark,
        SILVER_CONSTRUCTORS_TABLE
    )

    (
        target.alias("target")
        .merge(
            constructor_candidates_df.alias("source"),
            """
            target.constructor_id = source.constructor_id
            """
        )
        .whenMatchedUpdateAll(
            condition="""
                target.record_hash <> source.record_hash
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        f"Merged data into {SILVER_CONSTRUCTORS_TABLE}"
    )

else: 

    (
        constructor_candidates_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(SILVER_CONSTRUCTORS_TABLE)
    )

    print(f"Created {SILVER_CONSTRUCTORS_TABLE}")
